# Extended LLM arena: generate and critique questions, compare several models, blind-judge answers, revise weaker answers, and rank the field.

In [ ]:
import asyncio
import json
import os
import random
import statistics
import time
from pathlib import Path
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv(Path("../../../.env"), override=True)
random.seed(7)
DRY_RUN = False


## Setup: configure OpenAI-compatible providers, contestants, repetitions, and the judge; set `DRY_RUN = False` for a live run.

In [ ]:
PROVIDERS = {
    "openai": ("OPENAI_API_KEY", "https://api.openai.com/v1"),
    "anthropic": ("ANTHROPIC_API_KEY", "https://api.anthropic.com/v1/"),
    "gemini": ("GOOGLE_API_KEY", "https://generativelanguage.googleapis.com/v1beta/openai/"),
    "deepseek": ("DEEPSEEK_API_KEY", "https://api.deepseek.com"),
    "groq": ("GROQ_API_KEY", "https://api.groq.com/openai/v1"),
    "grok": ("GROK_API_KEY", "https://api.x.ai/v1"),
    "openrouter": ("OPENROUTER_API_KEY", "https://openrouter.ai/api/v1"),
    "ollama": (None, "http://localhost:11434/v1"),
}
CLIENTS = {} if DRY_RUN else {
    name: AsyncOpenAI(
        api_key="ollama" if name == "ollama" else os.getenv(key), base_url=url
    )
    for name, (key, url) in PROVIDERS.items()
}
CONTESTANTS = [
    {"model": "gpt-5.4-nano", "provider": "openai", "kwargs": {"reasoning_effort": "none"}},
    {"model": "claude-sonnet-4-6", "provider": "anthropic", "kwargs": {}},
    {"model": "gemini-3.1-flash-lite", "provider": "gemini", "kwargs": {}},
    {"model": "deepseek-v4-flash", "provider": "deepseek", "kwargs": {}},
    {"model": "openai/gpt-oss-120b", "provider": "groq", "kwargs": {}},
    {"model": "moonshotai/kimi-k2.6", "provider": "openrouter", "kwargs": {}},
    {"model": "gemma4:latest", "provider": "ollama", "kwargs": {}},
    {"model": "gpt-oss:20b", "provider": "ollama", "kwargs": {}},
]
BY_MODEL = {c["model"]: c for c in CONTESTANTS}
SEMAPHORES = {name: asyncio.Semaphore(1 if name == "ollama" else 4) for name in PROVIDERS}
JUDGE_MODEL = "grok-4.7"
RUBRIC = ["insight", "accuracy", "brevity", "nuance"]
REPS = 2
JUDGE_PASSES = 3


## Shared call helper: use canned outputs in dry-run mode or the same asynchronous chat API for every live provider.

In [ ]:
async def ask(provider, model, prompt, mock, **kwargs):
    async with SEMAPHORES[provider]:
        if DRY_RUN:
            return mock if isinstance(mock, str) else json.dumps(mock)
        response = await CLIENTS[provider].chat.completions.create(
            model=model, messages=[{"role": "user", "content": prompt}], **kwargs
        )
    return response.choices[0].message.content


## Stage 1: ask an orchestrator for varied, challenging questions that require short, nuanced answers.

In [ ]:
CATEGORIES = ["ethics", "epistemics", "ambiguity", "second-order effects", "practical wisdom", "self-reference"]
question_prompt = f"""Write {len(CATEGORIES)} hard LLM benchmark questions, one per category: {', '.join(CATEGORIES)}.
Require short answers that reward insight, accuracy, brevity, and nuance, not trivia.
Return ONLY a JSON array of objects with "category" and "question"."""
mock_questions = [{"category": c, "question": f"When should an AI defer to a human about {c}?"} for c in CATEGORIES]
questions = json.loads(await ask("openai", "gpt-6-luna", question_prompt, mock_questions))
for i, q in enumerate(questions, 1):
    q["id"] = f"q{i}"
for q in questions:
    print(f"[{q['id']}] ({q['category']}) {q['question']}")


## Stage 2: use Claude as a question critic so weak prompts are removed before the expensive fan-out.

In [ ]:
kept_questions = []
for q in questions:
    critic_prompt = f"""Question: {q['question']}
Keep it only if it is answerable, non-trivial, and rewards insight, accuracy, brevity, and nuance.
Return ONLY JSON: {{"keep": true, "reason": "one short sentence"}}."""
    verdict = json.loads(await ask("anthropic", "claude-sonnet-4-6", critic_prompt, {"keep": True, "reason": "Good dry-run question"}))
    print(f"[{q['id']}] keep={verdict['keep']}: {verdict['reason']}")
    if verdict["keep"]:
        q["critic_reason"] = verdict["reason"]
        kept_questions.append(q)
print(f"Kept {len(kept_questions)} of {len(questions)} questions.")


## Stage 3: collect a reference answer and grading criteria for each surviving question.

In [ ]:
REFERENCE_FORMAT = {"type": "json_schema", "json_schema": {
    "name": "reference_answer", "strict": True,
    "schema": {
        "type": "object",
        "properties": {"answer": {"type": "string"}, "criteria": {"type": "array", "items": {"type": "string"}}},
        "required": ["answer", "criteria"], "additionalProperties": False,
    },
}}
reference_answers = {}
for q in kept_questions:
    reference_prompt = f"""Write a short gold-standard answer and 3-4 grading criteria.
Question: {q['question']}
Return ONLY JSON: {{"answer": "...", "criteria": ["..."]}}."""
    mock_reference = {"answer": "Defer when human accountability and context matter.", "criteria": ["Accurate", "Concise", "Nuance"]}
    reference_answers[q["id"]] = json.loads(await ask("anthropic", "claude-sonnet-4-6", reference_prompt, mock_reference, response_format=REFERENCE_FORMAT))
    print(f"[{q['id']}] {reference_answers[q['id']]['answer']}")


## Stage 4: fan out every question to every contestant twice, with per-provider concurrency limits.

In [ ]:
async def answer_one(c, q, rep):
    started = time.perf_counter()
    text = await ask(
        c["provider"], c["model"], q["question"],
        f"Mock answer from {c['model']} to {q['id']}, repetition {rep}.",
        **c["kwargs"],
    )
    return {
        "model": c["model"], "provider": c["provider"], "question_id": q["id"],
        "rep": rep, "answer": text, "latency_s": time.perf_counter() - started,
    }

jobs = [
    answer_one(c, q, rep)
    for c in CONTESTANTS
    for q in kept_questions
    for rep in range(REPS)
]
answers = await asyncio.gather(*jobs)
print(f"Collected {len(answers)} answers from {len(CONTESTANTS)} contestants.")


## Stage 5: shuffle and anonymize answers, then have Grok score each answer three times against the rubric and reference.

In [ ]:
async def judge_pass(q, q_answers, pass_idx):
    shuffled = list(q_answers)
    random.shuffle(shuffled)
    labels = [chr(ord("A") + i) for i in range(len(shuffled))]
    label_map = dict(zip(labels, shuffled))
    block = "\n\n".join(f"Answer {label}:\n{a['answer']}" for label, a in zip(labels, shuffled))
    reference = reference_answers[q["id"]]
    prompt = f"""Judge these anonymous answers without guessing their authors.
Question: {q['question']}
Reference answer: {reference['answer']}
Grading criteria: {json.dumps(reference['criteria'])}
Score every answer from 1-10 on {', '.join(RUBRIC)} and add a one-sentence critique.
{block}
Return ONLY JSON shaped like {{"A": {{"insight": 8, "accuracy": 8, "brevity": 8, "nuance": 8, "critique": "..."}}}}."""
    mock_scores = {
        label: {**{d: 7 + i % 3 for d in RUBRIC}, "critique": "Be more specific."}
        for i, label in enumerate(labels)
    }
    scores = json.loads(await ask("grok", JUDGE_MODEL, prompt, mock_scores))
    return [
        {
            "model": label_map[label]["model"], "rep": label_map[label]["rep"],
            "question_id": q["id"], "pass": pass_idx,
            "scores": {d: values[d] for d in RUBRIC}, "critique": values["critique"],
        }
        for label, values in scores.items()
    ]

by_question = {}
for a in answers:
    by_question.setdefault(a["question_id"], []).append(a)
judge_jobs = [
    judge_pass(q, by_question[q["id"]], p)
    for q in kept_questions
    for p in range(JUDGE_PASSES)
]
judged = await asyncio.gather(*judge_jobs)
score_records = [record for group in judged for record in group]
print(f"Collected {len(score_records)} blind scores across {JUDGE_PASSES} passes.")


## Stage 6: send critiques to the bottom half, revise their answers, and blind-rejudge the full field to measure improvement.

In [ ]:
def mean_totals(records):
    totals = {}
    for r in records:
        totals.setdefault(r["model"], []).append(sum(r["scores"].values()))
    return {model: statistics.mean(values) for model, values in totals.items()}

current_totals = mean_totals(score_records)
bottom_models = set(sorted(current_totals, key=current_totals.get)[:max(1, len(current_totals) // 2)])
latest_critique = {(r["model"], r["question_id"], r["rep"]): r["critique"] for r in score_records}

async def revise_one(a):
    c = BY_MODEL[a["model"]]
    q = next(q for q in kept_questions if q["id"] == a["question_id"])
    critique = latest_critique[(a["model"], a["question_id"], a["rep"])]
    prompt = f"""Question: {q['question']}
Previous answer: {a['answer']}
Judge critique: {critique}
Revise the answer using this feedback and keep it short."""
    mock_revision = f"Improved mock answer from {a['model']} to {a['question_id']}."
    text = await ask(c["provider"], c["model"], prompt, mock_revision, **c["kwargs"])
    return dict(a, answer=text)

revised = await asyncio.gather(*(revise_one(a) for a in answers if a["model"] in bottom_models))
revised_lookup = {(a["model"], a["question_id"], a["rep"]): a for a in revised}
rejudge_jobs = [
    judge_pass(q, [revised_lookup.get((a["model"], a["question_id"], a["rep"]), a) for a in by_question[q["id"]]], 100 + p)
    for q in kept_questions
    for p in range(JUDGE_PASSES)
]
rejudged = await asyncio.gather(*rejudge_jobs)
revision_records = [record for group in rejudged for record in group]
new_totals = mean_totals(revision_records)
for model in sorted(bottom_models):
    print(f"{model}: {current_totals[model]:.1f} -> {new_totals[model]:.1f} ({new_totals[model] - current_totals[model]:+.1f})")


## Stage 7: rank the original answers and summarize repetition variance, provider latency, and judge consistency.

In [ ]:
per_model = {}
answer_pass_totals = {}
for r in score_records:
    entry = per_model.setdefault(r["model"], {d: [] for d in RUBRIC})
    for d in RUBRIC:
        entry[d].append(r["scores"][d])
    key = (r["model"], r["question_id"], r["rep"])
    answer_pass_totals.setdefault(key, []).append(sum(r["scores"].values()))

rep_means = {key: statistics.mean(values) for key, values in answer_pass_totals.items()}
rep_groups = {}
for (model, qid, rep), total in rep_means.items():
    rep_groups.setdefault((model, qid), []).append(total)
variance_groups = {}
for (model, qid), totals in rep_groups.items():
    variance_groups.setdefault(model, []).append(statistics.variance(totals))
rep_variance = {model: statistics.mean(values) for model, values in variance_groups.items()}

rows = []
for model, dimensions in per_model.items():
    means = {d: statistics.mean(values) for d, values in dimensions.items()}
    rows.append({"model": model, **means, "total": sum(means.values()), "rep_variance": rep_variance[model]})
rows.sort(key=lambda row: row["total"], reverse=True)
for rank, row in enumerate(rows, 1):
    dimensions = " ".join(f"{d}={row[d]:.1f}" for d in RUBRIC)
    print(f"#{rank} {row['model']}: {dimensions} total={row['total']:.1f} rep_var={row['rep_variance']:.2f}")

latencies = {}
for a in answers:
    latencies.setdefault(a["provider"], []).append(a["latency_s"])
for provider, values in sorted(latencies.items()):
    print(f"{provider}: mean latency {statistics.mean(values):.3f}s")
consistency = statistics.mean(statistics.stdev(values) for values in answer_pass_totals.values())
print(f"Judge self-consistency: mean per-answer stdev={consistency:.2f} (lower is steadier)")


## Exercise: identify the orchestrator, evaluator, and evaluator-optimizer patterns, then add a contestant or change the rubric to compare quality, speed, and consistency.